<header>
   <p style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       RAG Teradata VectorStore Embeddings Explorer
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style='font-size:20px;font-family:Arial;color:#00233C'><b>Introduction</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Imagine you are a data scientist at a large enterprise. Your team has a vast knowledge base documents, facts, records - all sitting inside <b>Teradata Vantage</b>. 
Business users keep asking questions, but traditional keyword search keeps failing them. They need answers grounded in <i>meaning</i>, not just matching words.
</p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
This demo walks you through building a <b>Retrieval-Augmented Generation (RAG)</b> pipeline that solves exactly this problem — entirely inside Teradata Vantage, 
powered by enterprise-grade vector search and a large language model routed through your organization's secure <b>LiteLLM proxy</b>.
</p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
By the end of this notebook, you will have a working system that can take any natural-language question, retrieve the most semantically relevant documents from Vantage, 
and generate a grounded, accurate answer — no hallucinations, no guesswork.
</p>

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>How It Works — The RAG Journey</b></p>

<ul style='font-size:16px;font-family:Arial;color:#00233C'>
    <li><b>Step 1 — Store:</b> Documents are embedded using Amazon Titan (via Teradata's native Bedrock integration) and stored as vectors inside Teradata Vantage.</li>
    <li><b>Step 2 — Retrieve:</b> When a question arrives, Vantage performs a lightning-fast semantic similarity search to find the most relevant documents.</li>
    <li><b>Step 3 — Augment:</b> The retrieved documents are injected as grounding context into a carefully engineered prompt.</li>
    <li><b>Step 4 — Generate:</b> The prompt is sent to Claude (via the LiteLLM proxy), which generates a precise, context-grounded answer.</li>
</ul>

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>Key Components</b></p>

<ul style='font-size:16px;font-family:Arial;color:#00233C'>
    <li><b>Teradata Enterprise Vector Store:</b> Stores and searches high-dimensional embedding vectors natively inside Vantage. No external vector database needed — your data never leaves your enterprise environment.</li>
    <li><b>Amazon Titan Embed (<code>amazon.titan-embed-text-v2:0</code>):</b> Called directly by Vantage via UES to convert document text into semantic vectors at indexing time.</li>
    <li><b>LiteLLM Proxy:</b> A centralized, credential-free gateway that routes all LLM calls (Claude) through your organization's secure infrastructure.</li>
    <li><b>LangChain + ChatOpenAI:</b> Provides the orchestration layer — retriever, prompt template, and LLM invocation — wired together into a clean RAG closure.</li>
</ul>

<hr style='height:2px;border:none'>

<b style='font-size:20px;font-family:Arial;color:#00233C'>1. Configuring the Environment</b>

<div class="alert alert-block alert-info">
<p style='font-size:16px;font-family:Arial;'><b>Note: </b><i>The installation of the required libraries will take approximately <b>3 to 5 minutes</b> on first run. 
If the libraries are already installed, this completes in seconds.</i></p>
</div>


In [ ]:
%%capture
%pip install -r requirements.txt --quiet

<div class="alert alert-block alert-info">
<p style='font-size:16px;font-family:Arial;'><b>Note: </b><i>Please restart the kernel after the installation above before proceeding. 
The simplest way to restart the Kernel is by typing zero zero: <b>0 0</b>.</i></p>
</div>

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>1.1 Import the Required Libraries</b></p>

<p style = 'font-size:16px;font-family:Arial'>Here, we import the required libraries, set environment variables and environment paths (if required).</p>

In [ ]:
import numpy as np
import pandas as pd
import os
import re
import json
import warnings
import sys
warnings.filterwarnings('ignore')

import panel as pn
import litellm
from litellm import completion
from dotenv import load_dotenv, dotenv_values
from teradatagenai import VSManager
from langchain_core.documents import Document
from langchain_teradata import TeradataVectorStore
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from teradataml import *
import teradataml as tdml
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pd.set_option('display.max_colwidth', None)
display.max_rows = 5

<hr style='height:2px;border:none'>
<p style='font-size:20px;font-family:Arial;color:#00233C'><b>2. Connecting to VantageCloud Lake</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Our first destination is <b>Teradata VantageCloud Lake</b> — the enterprise analytics platform where all our data lives and where the vector store will be created.
We load connection credentials securely from a local <code>.env</code> file, so no sensitive information is ever hardcoded in the notebook.
</p>

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>2.1 Establish the Vantage Connection</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
We read the host, username, and password from the environment file and create a Vantage session using <code>create_context</code>.
A query band is set to tag all SQL traffic from this demo for monitoring and governance purposes.
</p>


In [ ]:
ENV_PATH = "/home/jovyan/JupyterLabRoot/VantageCloud_Lake/.config/.env"
if os.path.exists(ENV_PATH):
    print("Environment parameter file found. Proceeding with connection...")
    env_vars = dotenv_values(ENV_PATH)
    # Create Vantage context
    eng = create_context(
        host     = env_vars.get("host"),
        username = env_vars.get("username"),
        password = env_vars.get("my_variable"),
    )
    execute_sql('''SET query_band='DEMO=RAG_VectorStore_Embeddings_Explorer.ipynb;' UPDATE FOR SESSION;''')
    print("Connected to VantageCloud Lake with:", eng)
else:
    print("Environment file not found. Please contact the support team.")
    sys.exit(1)

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>2.2 UES Authentication — Unlocking the Vector Store</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Teradata's <b>Unified Experience Services (UES)</b> acts as the secure gateway between Vantage and external AI services like Amazon Bedrock. 
Without a valid UES token, the Vector Store cannot call the embedding model. 
Think of this as showing your badge at the door before accessing the AI wing of the building.
</p>


In [ ]:
ues_uri = env_vars.get("ues_uri", "")
if ues_uri.endswith("/open-analytics"):
    ues_uri = ues_uri[:-15]
if set_auth_token(
    base_url  = ues_uri,
    pat_token = env_vars.get("access_token"),
    pem_file  = env_vars.get("pem_file"),
):
    print(" UES Authentication successful")
else:
    print(" UES Authentication failed. Check credentials.")
    sys.exit(1)

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>2.3 Verify the Vector Store Service</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
A quick health check confirms that Teradata's Vector Store engine is up and reachable. 
If this passes, we are cleared for takeoff.
</p>


In [ ]:
VSManager.health()

<hr style='height:2px;border:none'>
<p style='font-size:20px;font-family:Arial;color:#00233C'><b>3. Configuring the LiteLLM Proxy</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
In an enterprise setting, direct API calls to cloud LLMs are often restricted for security and cost governance reasons.
The <b>LiteLLM proxy</b> solves this elegantly — it acts as a centralized router that handles authentication, 
rate limiting, and model routing on behalf of all applications.
</p>

<ul style='font-size:16px;font-family:Arial;color:#00233C'>
    <li><b>LITELLM_LOG:</b> Enables detailed debug-level logging for request tracing and troubleshooting.</li>
    <li><b>LITELLM_API_KEY:</b> The single key that authenticates us to the proxy — loaded securely from the <code>.env</code> file.</li>
    <li><b>LITELLM_BASE_URL:</b> The proxy endpoint. All LLM requests are routed here instead of directly to the cloud provider.</li>
</ul>


In [ ]:
os.environ["LITELLM_LOG"] = "DEBUG"

In [ ]:
load_dotenv("/home/jovyan/JupyterLabRoot/VantageCloud_Lake/.config/.env")
os.environ["LITELLM_API_KEY"] = os.getenv("litellm_key")
litellm_key  = os.environ["LITELLM_API_KEY"]
litellm_base = os.getenv("litellm_base_url")

In [ ]:
if os.environ.get('LITELLM_API_KEY') and litellm_base:
    print(" LiteLLM configured successfully")
    print(f"  Base URL: {litellm_base}")
    print(f"  API Key : {litellm_key[:10]}...")
else:
    print(" Error: LiteLLM credentials not found in .env file")
    print("  Please check /home/jovyan/JupyterLabRoot/VantageCloud_Lake/.config/.env")

<hr style='height:2px;border:none'>
<p style='font-size:20px;font-family:Arial;color:#00233C'><b>4. Building the Knowledge Base</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Every RAG system needs a knowledge base — a collection of documents that ground the LLM's answers in facts.
In this demo, we define six short documents covering topics like LangChain, RAG, BM25, vector search, geography, and Teradata itself.
</p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
In a real-world scenario, these documents could come from a Teradata table, a data warehouse, 
a document repository, or any other enterprise data source. The pipeline is the same — 
only the source changes.
</p>

<div class="alert alert-block alert-info">
<p style='font-size:16px;font-family:Arial;'><b>Note: </b><i>These documents are intentionally short and diverse to make the PCA vector visualization in Section 7 more visually interesting and interpretable.</i></p>
</div>


In [ ]:
docs = [
    Document(page_content="LangChain is a framework for building LLM applications."),
    Document(page_content="RAG means Retrieval-Augmented Generation."),
    Document(page_content="BM25 is a keyword-based retrieval algorithm."),
    Document(page_content="Vector databases are useful for semantic search."),
    Document(page_content="Paris is the capital of France."),
    Document(page_content="Teradata Vantage is a powerful cloud analytics platform."),
]
print(f"Loaded {len(docs)} documents into the knowledge base.")

<hr style='height:2px;border:none'>
<p style='font-size:20px;font-family:Arial;color:#00233C'><b>5. Creating the Teradata Vector Store</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
This is where the magic happens. We hand our documents to <b>Teradata's Enterprise Vector Store</b>, 
which internally calls <b>Amazon Bedrock</b> (via UES) to embed each document into a high-dimensional numeric vector. 
These vectors capture the <i>semantic meaning</i> of each document and are stored persistently inside Vantage.
</p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Unlike external vector databases, everything lives <b>inside Teradata Vantage</b> — 
giving you enterprise-grade security, data governance, and scalability with zero additional infrastructure.
</p>

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>5.1 Initialize and Recreate the Vector Store</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
We first check if a vector store with this name already exists and destroy it if so — 
making the notebook safe to re-run from scratch without manual cleanup.
</p>


In [ ]:
VECTORSTORE_NAME = env_vars.get("username") + '_rag_vs'

Destroy existing store if it exists

In [ ]:
try:
    _vs_check = TeradataVectorStore(name=VECTORSTORE_NAME)
    if isinstance(_vs_check.status(), pd.core.frame.DataFrame):
        _vs_check.destroy()
        print(f"Existing vector store '{VECTORSTORE_NAME}' destroyed.")
except Exception:
    pass  # store did not exist yet — that's fine

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Now we create the vector store. Vantage will embed each document and index the resulting vectors 
for fast similarity search. This is the step that turns raw text into a searchable semantic index.
</p>


In [ ]:
vectorstore = TeradataVectorStore.from_documents(
    name             = VECTORSTORE_NAME,
    documents        = docs,
    embeddings_model = 'amazon.titan-embed-text-v2:0',
)
print(f" Vector store '{VECTORSTORE_NAME}' created in Teradata Vantage")
print(f"  Documents indexed: {len(docs)}")

<hr style='height:2px;border:none'>
<p style='font-size:20px;font-family:Arial;color:#00233C'><b>6. Wiring Up the LLM</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
With our knowledge base ready inside Vantage, we now connect our language model — 
<b>Claude Sonnet</b> running on AWS Bedrock, accessed securely through the <b>LiteLLM proxy</b>.
</p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Because the LiteLLM proxy exposes an <b>OpenAI-compatible REST interface</b>, 
we can use LangChain's familiar <code>ChatOpenAI</code> wrapper and simply redirect 
<code>base_url</code> to our proxy. No vendor-specific SDK required.
</p>

In [ ]:
llm = ChatOpenAI(
    model       = "claude-sonnet-4-6",
    api_key     = litellm_key,
    base_url    = litellm_base,
    temperature = 0,
    default_headers = {"x-litellm-routing": "proxy"},
)

<hr style='height:2px;border:none'>
<p style='font-size:20px;font-family:Arial;color:#00233C'><b>7. The RAG Pipeline — Putting It All Together</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Here is where the two halves of our system meet. The <code>build_rag_answer</code> function 
assembles the full Retrieval-Augmented Generation pipeline as a reusable closure:
</p>

<ol style='font-size:16px;font-family:Arial;color:#00233C'>
    <li>A question arrives from the user.</li>
    <li>The <b>Teradata Vector Store retriever</b> converts the question to an embedding and fetches the top-k most semantically relevant documents from Vantage.</li>
    <li>Those documents are stitched together as <b>grounding context</b> and injected into a structured prompt.</li>
    <li>The prompt is sent to <b>Claude via the LiteLLM proxy</b>, which generates an answer using only the provided context — no hallucinations.</li>
    <li>The answer is returned together with the source documents that supported it.</li>
</ol>

<div class="alert alert-block alert-info">
<p style='font-size:16px;font-family:Arial;'><b>Note: </b><i>The system prompt enforces strict grounding rules — the model will explicitly say 
"I don't know based on the provided documents" rather than inventing an answer when the context is insufficient.</i></p>
</div>


In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant answering questions using only the provided context.
Rules:
- If the context contains relevant information, answer using that information, even if the wording of the question is indirect.
- You may restate or lightly infer from the context, but do not invent facts not supported by it.
- If the context does not contain enough information to answer, say:
  "I don't know based on the provided documents."
- When possible, answer directly and briefly.
"""
def build_rag_answer(vectorstore, llm, k: int = 4, return_sources: bool = False):
    """
    Build a closure that, given a question, retrieves context from the
    Teradata vector store and generates a grounded answer via the LLM.
    Parameters
    ----------
    vectorstore   : TeradataVectorStore  – the populated vector store
    llm           : ChatOpenAI           – LangChain-compatible LLM instance
    k             : int                  – number of documents to retrieve
    return_sources: bool                 – if True, also return source text
    Returns
    -------
    Callable[[str], str | tuple[str, str]]
    """
    retriever = vectorstore.as_retriever(
        search_type   = "similarity",
        search_kwargs = {"top_k": k},
    )
    prompt = ChatPromptTemplate.from_template("""
<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>
Answer the question using only the context below.
If the context contains partial but relevant information, use it to give the best grounded answer.
If the context does not contain enough information, say:
"I don't know based on the provided documents."
Context:
{context}
Question:
{question}
[/INST]
""")
    def rag_answer(question: str):
        retrieved_docs = retriever.invoke(question)
        context        = "\n\n".join(doc.page_content for doc in retrieved_docs)
        messages  = prompt.format_messages(
            system_prompt = SYSTEM_PROMPT,
            context       = context,
            question      = question,
        )
        response  = llm.invoke(messages)
        answer    = response.content if hasattr(response, "content") else str(response)
        if not return_sources:
            return answer
        sources_md = "\n\n".join(
            f"**Source {i+1}**\n\n{doc.page_content}"
            for i, doc in enumerate(retrieved_docs)
        ) or "No sources retrieved."
        return answer, sources_md
    return rag_answer
rag_answer = build_rag_answer(
    vectorstore    = vectorstore,
    llm            = llm,
    k              = 4,
    return_sources = True,
)
print(" RAG pipeline ready")

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>7.1 Chat Interface</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
With the RAG pipeline ready, we now launch an interactive <b>Panel chat interface</b> — 
so you can converse with your Teradata knowledge base in plain English, in real time,
directly inside this notebook.
</p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Every message you type triggers the full RAG loop: the question is embedded, 
the most relevant documents are retrieved from Vantage, and Claude generates a grounded answer.
</p>

<ul style='font-size:16px;font-family:Arial;color:#00233C'>
    <li>Try asking: <i>What is RAG?</i></li>
    <li>Try asking: <i>What is LangChain used for?</i></li>
    <li>Try asking: <i>What is the capital of France?</i></li>
    <li>Try asking something <b>outside</b> the knowledge base — the model should say it doesn't know rather than guess.</li>
</ul>

<div class="alert alert-block alert-info">
<p style='font-size:16px;font-family:Arial;'><b>Note: </b><i>The chatbot accesses the Teradata Vector Store and the LiteLLM proxy on every message. 
A brief delay on the first query is normal. Panel renders the chat interface natively inside the notebook — no external ports or links required.</i></p>
</div>

In [ ]:
pn.extension()

def callback(contents, user, instance):
    """Handles the chat interaction and returns the RAG response."""
    answer, sources_md = rag_answer(contents)
    return answer

pn.chat.ChatInterface(
    callback=callback,
    show_rerun=False,
    show_undo=False,
    show_clear=False,
    width=800,
    height=400
).servable()

<hr style='height:2px;border:none'>
<p style='font-size:20px;font-family:Arial;color:#00233C'><b>8. Under the Hood — Exploring the Embedding Vectors</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
So far we have experienced RAG from the outside — questions in, answers out. 
But what is actually happening inside Teradata Vantage? Let's pull back the curtain and inspect the embedding vectors directly.
</p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Each document has been transformed into a list of hundreds of floating-point numbers — 
a coordinate in a high-dimensional semantic space where <b>similar meanings cluster together</b>. 
We will retrieve these raw vectors from Vantage, measure similarity between documents, 
and project everything down to 2-D so we can actually <i>see</i> how the model has organized our knowledge base.
</p>

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>8.1 Retrieve Raw Vectors from Vantage</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
We call <code>get_details()</code> and <code>get_indexes_embeddings()</code> to inspect the vector store metadata 
and the binary-encoded embedding vectors stored inside Vantage.
</p>


In [ ]:
details            = vectorstore.get_details()
indexes_embeddings = vectorstore.get_indexes_embeddings()
details

In [ ]:
indexes_embeddings

Check vector dimension from the first stored vector

In [ ]:
sample_value = indexes_embeddings.to_pandas(num_rows=1).vector_index.values[0]
vector_shape = np.frombuffer(sample_value, dtype=np.float32).shape

print(f"Vector shape (first document): {vector_shape}")

num_vectors = indexes_embeddings.shape[0]
vector_dim  = vector_shape[0] if num_vectors > 0 else 0

print(f"\nNumber of demo vectors : {num_vectors}")
print(f"Vector dimension       : {vector_dim}")

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>8.2 Decode the Binary Vectors</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Vantage stores vectors as binary blobs for efficiency. We decode them back into NumPy float arrays 
and attach shortened document labels for our visualizations.
</p>


In [ ]:
vector_data = indexes_embeddings.to_pandas()

Decode binary blobs back to float arrays

In [ ]:
vector_data['vector_index'] = vector_data['vector_index'].apply(
    lambda x: np.frombuffer(x, dtype=np.float64)
)

vector_data['vector_index_normalized'] = vector_data['vector_index_normalized'].apply(
    lambda x: np.frombuffer(x, dtype=np.float64)
)

labels = [doc[:50].replace("\n", " ") for doc in vector_data.text.values]
vector_data.head(1)

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>8.3 Cosine Similarity — How Close Are Two Documents?</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Cosine similarity measures the angle between two vectors in high-dimensional space. 
A score of <b>1.0</b> means identical meaning; <b>0.0</b> means completely unrelated.
We compute this between the first two documents to get a feel for how the embedding model has placed them relative to each other.
</p>


In [ ]:
query_1 = vector_data.text.values[0]
query_2 = vector_data.text.values[1]

q1_vec = vector_data.vector_index.values[0].reshape(1, -1)
q2_vec = vector_data.vector_index.values[1].reshape(1, -1)

query_similarity = cosine_similarity(q1_vec, q2_vec)[0][0]

print(f'Query 1 : "{query_1}"')
print(f'Query 2 : "{query_2}"')

print(f'Cosine similarity : {query_similarity:.4f}')

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>8.4 PCA — Projecting into 2-D</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Embedding vectors have hundreds of dimensions — impossible to visualize directly. 
<b>Principal Component Analysis (PCA)</b> compresses them to 2-D while preserving as much variance 
(i.e. meaningful spread) as possible. Documents that are semantically similar in the original 
high-dimensional space should appear close together in the projection.
</p>


In [ ]:
doc_vectors = vector_data.vector_index.tolist()
all_vectors = np.vstack([doc_vectors, q1_vec, q2_vec])
pca    = PCA(n_components=2)
all_2d = pca.fit_transform(all_vectors)
doc_2d = all_2d[:num_vectors]
q1_2d  = all_2d[num_vectors]
q2_2d  = all_2d[num_vectors + 1]
print(f"Explained variance ratio (PC1, PC2): {pca.explained_variance_ratio_.round(3)}")

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>8.5 Matplotlib Visualization — The Semantic Map</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
The chart below is a <b>semantic map</b> of our knowledge base. Each arrow points from the origin 
to a document's position in the 2-D semantic space. Documents on similar topics cluster together; 
the two highlighted queries (<span style='color:#E87722;'>Query 1</span> and <span style='color:#00A9CE;'>Query 2</span>) 
show where your questions land relative to the documents — which is exactly what the retriever uses to find the best matches.
</p>


In [ ]:
get_ipython().run_line_magic('matplotlib', 'inline')

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(doc_2d[:, 0], doc_2d[:, 1], s=80, color='#00233C', label="Documents", zorder=3)

for i, label in enumerate(labels):
    x, y = doc_2d[i]
    ax.annotate("", xy=(x, y), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color='#00233C', alpha=0.25))
    ax.text(x, y + 0.01, label, fontsize=8, color='#00233C')
ax.scatter(q1_2d[0], q1_2d[1], s=140, marker='X', color='#E87722', label="Query 1", zorder=4)
ax.scatter(q2_2d[0], q2_2d[1], s=140, marker='X', color='#00A9CE', label="Query 2", zorder=4)

for qvec, qcolor in [(q1_2d, '#E87722'), (q2_2d, '#00A9CE')]:
    ax.annotate("", xy=(qvec[0], qvec[1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=qcolor, lw=2, alpha=0.8))
    
ax.axhline(0, linewidth=0.5, color='gray')
ax.axvline(0, linewidth=0.5, color='gray')
ax.set_title("Document and Query Embeddings (PCA 2-D Projection)",
             fontsize=14, fontfamily='DejaVu Sans', color='#00233C')
ax.set_xlabel("Principal Component 1", fontfamily='DejaVu Sans', color='#00233C')
ax.set_ylabel("Principal Component 2", fontfamily='DejaVu Sans', color='#00233C')
ax.legend()
plt.tight_layout()
plt.show()

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>8.6 Interactive Plotly Visualization</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
The static chart gives us the big picture. The interactive Plotly version below lets you 
<b>hover over each point</b> to see the full document text, exact PCA coordinates, and point type. 
Use this to intuitively explore which documents are semantically close — 
and understand why the retriever returns the results it does.
</p>


In [ ]:
pio.renderers.default = "notebook_connected"
plot_df = pd.DataFrame({
    "x"    : np.concatenate([doc_2d[:, 0], [q1_2d[0], q2_2d[0]]]),
    "y"    : np.concatenate([doc_2d[:, 1], [q1_2d[1], q2_2d[1]]]),
    "label": labels + [query_1, query_2],
    "type" : ["Document"] * num_vectors + ["Query", "Query"],
})
fig = px.scatter(
    plot_df,
    x          = "x",
    y          = "y",
    text       = "label",
    color      = "type",
    color_discrete_map = {"Document": "#00233C", "Query": "#E87722"},
    title      = "Interactive PCA Projection of Document & Query Embeddings",
    hover_data = {"x": ':.3f', "y": ':.3f', "label": True, "type": True},
)
fig.update_traces(textposition="top center")
fig.update_layout(
    width  = 900,
    height = 620,
    font   = dict(family="Arial", color="#00233C"),
)
fig.show(renderer="notebook")

<p style='font-size:18px;font-family:Arial;color:#00233C'><b>8.7 Similarity Search — Seeing What the Retriever Sees</b></p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
Finally, let's call <code>similarity_search</code> directly and inspect the raw retrieval results.
This is what happens inside the RAG pipeline every time a question is asked — 
Vantage finds the top-k documents whose embedding vectors are closest to the query vector.
</p>

<p style='font-size:16px;font-family:Arial;color:#00233C'>
We convert Euclidean distance to an intuitive similarity score using:
</p>

$$similarity = \frac{1}{1 + distance}$$


In [ ]:
query   = "What is RAG?"
results = vectorstore.similarity_search(
    question    = query,
    top_k       = 3,
    return_type = "json",
)
print(f"Top matches for: '{query}'\n")
if isinstance(results, list):
    for i, row in enumerate(results, 1):
        print(f"Result {i}")
        if isinstance(row, Document):
            print(row.page_content)
        elif isinstance(row, dict):
            for key in ("score", "similarity", "distance", "content",
                        "page_content", "text", "document"):
                if key in row:
                    print(f"  {key} = {row[key]}")
            if row.get("metadata"):
                print(f"  metadata = {row['metadata']}")
        else:
            print(row)
        print("-" * 60)
else:
    print("Unexpected response type:", type(results))
    print(results)

<hr style='height:2px;border:none'>
<b style='font-size:20px;font-family:Arial;color:#00233C'>9. Cleanup</b>

<p style = 'font-size:16px;font-family:Arial'>Call the destroy() method of the VS object to clean up the objects created during this demo.</p>

In [ ]:
vectorstore.destroy()

In [ ]:
df = document_vector_store.status()

while True:
    if df is None:
        break
    else:
        print(f"Current status: {df}. Waiting 10 seconds...")
        time.sleep(10)
        df = document_vector_store.status()

print(f"Vector store '{VECTORSTORE_NAME}' destroyed")

In [ ]:
VSManager.disconnect()

In [ ]:
remove_context()

<p style='font-size:16px;font-family:Arial;color:#00233C'><b>Links : </b></p>
<ul style='font-size:16px;font-family:Arial;color:#00233C'>
    <li>Teradata Enterprise Vector Store: <a href='https://docs.teradata.com/search/all?query=Teradata+Enterprise+Vector+Store&content-lang=en-US'>Official Documentation</a></li>
    <li>LangChain Teradata Integration: <a href='https://docs.teradata.com/r/Enterprise/Teradata-Package-for-LangChain-Function-Reference'>Teradata Package for LangChain</a></li>
</ul>

<footer style="padding-bottom:35px; border-bottom:3px solid #00233C;">
    <div style="float:left;margin-top:14px;font-family:Arial;color:#00233C;">ClearScape Analytics™</div>
    <div style="float:right;">
        <div style="float:left; margin-top:14px;font-family:Arial;color:#00233C;">
            Copyright © Teradata Corporation - 2026. All Rights Reserved
        </div>
    </div>
</footer>
